# 📊 고도화된 기하학적 시공간 물리 분석 일반화 검증 노트북 (V2 - holdout_300)

본 노트북은 비디오 프레임 순서 정렬 과제에서 **카메라 수평 패닝(Pan Left/Right)** 및 **객체 좌우 이동 물리 인과성 규칙**과 **장면 전환 검출 성능**을 실제 데이터셋 `splits/holdout_300.csv` 300개 전량을 대상으로 분석 및 정량 평가하기 위해 작성되었습니다.

---

In [ ]:
import os
import re
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
import torch.nn.functional as F

# PyTorch 중복 라이브러리 경고 방지
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

try:
    from transformers import OwlViTProcessor, OwlViTForObjectDetection
    HAS_OWL = True
    print('✅ OWL-ViT (transformers) 로드 성공!')
except ImportError:
    HAS_OWL = False
    print('⚠️ transformers 라이브러리가 없습니다. 데모용 Mock 클래스로 대체됩니다.')

try:
    from transformers import CLIPProcessor, CLIPModel
    HAS_CLIP = True
    print('✅ CLIP (transformers) 로드 성공!')
except ImportError:
    HAS_CLIP = False
    print('⚠️ CLIP 라이브러리가 없습니다. 데모용 Mock 클래스로 대체됩니다.')

from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

### 1. 텍스트 임베딩 기반 가중치 얼라이너 (Text Embedding Aligner)
캡션 문장의 뉘앙스에 따라 '공간적 깊이(Depth)'와 '평면적 이동(Trajectory)' 중 어떤 분석 도구를 더 신뢰할지 수학적으로 자동 가중치를 배분합니다.

In [ ]:
class TextEmbeddingAligner:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", device="cpu"):
        self.device = device
        self.has_model = False
        try:
            from transformers import AutoTokenizer, AutoModel
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name).to(self.device)
            self.model.eval()
            self.has_model = True
            print("✅ Sentence-Transformer 모델 로드 성공!")
        except Exception:
            print("ℹ️ 오프라인 상태 혹은 모델이 없습니다. Bag-of-Words 임베딩 폴백을 사용합니다.")
            
        self.depth_anchors = ["zooms in", "zooms out", "closer", "further", "distant", "approaching", "receding", "가까워짐", "멀어짐", "다가옴", "줌인", "줌아웃", "클로즈업"]
        self.trajectory_anchors = ["moves left", "moves right", "raises higher", "falls down", "shifts left", "shifts right", "오른쪽", "왼쪽", "위로", "아래로", "이동", "상하", "좌우"]

    def get_embedding(self, text):
        if self.has_model:
            inputs = self.tokenizer(text, padding=True, truncation=True, return_tensors="pt").to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            return embeddings / embeddings.norm(dim=-1, keepdim=True)
        else:
            vocab = list(set(self.depth_anchors + self.trajectory_anchors))
            words = text.lower().split()
            vec = np.zeros(len(vocab))
            for w in words:
                for idx, anchor in enumerate(vocab):
                    if anchor in w or w in anchor:
                        vec[idx] += 1.0
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec = vec / norm
            return torch.tensor([vec], dtype=torch.float32)

    def calculate_weights(self, caption):
        caption_vec = self.get_embedding(caption)
        depth_vec = self.get_embedding(" ".join(self.depth_anchors))
        traj_vec = self.get_embedding(" ".join(self.trajectory_anchors))
        
        cos_depth = F.cosine_similarity(caption_vec, depth_vec).item()
        cos_traj = F.cosine_similarity(caption_vec, traj_vec).item()
        
        exp_d = np.exp(cos_depth * 5.0)
        exp_t = np.exp(cos_traj * 5.0)
        sum_exp = exp_d + exp_t
        
        w_depth = exp_d / sum_exp
        w_traj = exp_t / sum_exp
        
        return w_depth, w_traj, cos_depth, cos_traj

### 2. OWL-ViT 기하학적 시공간 물리 분석기 (Spatial3DOwlViTTrajectoryExtractor)

In [ ]:
def extract_candidate_nouns(caption):
    clean_caption = re.sub(r'[^\w\s]', '', caption.lower())
    words = clean_caption.split()
    stopwords = {
        'a', 'an', 'the', 'in', 'on', 'at', 'with', 'by', 'to', 'of', 'for', 'and', 'or', 'but',
        'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
        'this', 'that', 'these', 'those', 'then', 'there', 'here', 'it', 'its', 'they', 'them', 'their',
        'he', 'him', 'his', 'she', 'her', 'we', 'us', 'our', 'you', 'your', 'moves', 'shifts', 'zooms',
        'closer', 'further', 'away', 'into', 'out', 'up', 'down', 'left', 'right'
    }
    candidates = [w for w in words if w not in stopwords and len(w) > 2]
    candidates = list(set(candidates))
    if 'person' not in candidates:
        candidates.append('person')
    return candidates

class Spatial3DOwlViTTrajectoryExtractor:
    def __init__(self, model_name='google/owlvit-base-patch32', device='cpu'):
        self.device = device
        if HAS_OWL:
            self.processor = OwlViTProcessor.from_pretrained(model_name)
            self.model = OwlViTForObjectDetection.from_pretrained(model_name).to(device)
            self.model.eval()
        else:
            self.processor = None
            self.model = None
        
    def extract_3d_spatial_features(self, image_paths, query_text):
        results_summary = []
        text_queries = [[query_text]]
        
        for idx, img_path in enumerate(image_paths):
            if not os.path.exists(img_path):
                results_summary.append({
                    'frame': idx + 1,
                    'status': 'file_not_found',
                    'bbox': None,
                    'center': None,
                    'area_ratio': 0.0,
                    'score': 0.0
                })
                continue
                
            img = Image.open(img_path).convert('RGB')
            w, h = img.size
            img_area = w * h
            
            if HAS_OWL:
                inputs = self.processor(text=text_queries, images=img, return_tensors='pt')
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                with torch.no_grad():
                    outputs = self.model(**inputs)
                target_sizes = torch.tensor([img.size[::-1]], dtype=torch.float32).to(self.device)
                results = self.processor.post_process_object_detection(
                    outputs=outputs, target_sizes=target_sizes, threshold=0.05
                )[0]
                boxes = results['boxes'].cpu().numpy()
                scores = results['scores'].cpu().numpy()
            else:
                # Mock output if OWL is not used
                boxes = np.array([[w*0.2, h*0.2, w*0.8, h*0.8]])
                scores = np.array([0.95])
            
            if len(boxes) == 0:
                results_summary.append({
                    'frame': idx + 1,
                    'status': 'missed_detection',
                    'bbox': None,
                    'center': None,
                    'area_ratio': 0.0,
                    'score': 0.0
                })
                continue
                
            best_idx = 0
            max_area = 0
            for i, box in enumerate(boxes):
                box_w = box[2] - box[0]
                box_h = box[3] - box[1]
                area = box_w * box_h
                if area > max_area:
                    max_area = area
                    best_idx = i
            
            best_box = boxes[best_idx]
            best_score = scores[best_idx] if len(scores) > best_idx else 0.5
            x1, y1, x2, y2 = map(int, best_box)
            cx = ((x1 + x2) / 2) / w
            cy = ((y1 + y2) / 2) / h
            best_area_ratio = max_area / img_area
            
            results_summary.append({
                'frame': idx + 1,
                'status': 'success',
                'bbox': (x1, y1, x2, y2),
                'center': (cx, cy),
                'area_ratio': best_area_ratio,
                'score': best_score
            })
            
        return results_summary

    def select_best_query(self, image_paths, candidate_queries):
        best_query = 'person'
        max_avg_score = -1.0
        query_scores = {}
        
        for q in candidate_queries:
            features = self.extract_3d_spatial_features(image_paths, q)
            scores = [f['score'] for f in features if f['status'] == 'success']
            avg_score = sum(scores) / len(image_paths) if len(scores) > 0 else 0.0
            
            query_scores[q] = avg_score
            if avg_score > max_avg_score:
                max_avg_score = avg_score
                best_query = q
                
        return best_query, query_scores

### 3. CLIP 장면 divider (CLIPSceneDivider)

In [ ]:
class CLIPSceneDivider:
    def __init__(self, model_name='openai/clip-vit-base-patch32', device='cpu'):
        self.device = device
        if HAS_CLIP:
            self.processor = CLIPProcessor.from_pretrained(model_name)
            self.model = CLIPModel.from_pretrained(model_name).to(device)
            self.model.eval()
        else:
            self.processor = None
            self.model = None
            
    def analyze_cuts(self, image_paths, threshold=0.85):
        scene_groups = []
        similarities = []
        
        imgs = []
        for path in image_paths:
            if os.path.exists(path):
                imgs.append(Image.open(path).convert('RGB'))
            else:
                imgs.append(Image.new('RGB', (100, 100)))
                
        if HAS_CLIP:
            inputs = self.processor(images=imgs, return_tensors='pt').to(self.device)
            with torch.no_grad():
                image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            
            for i in range(len(imgs) - 1):
                sim = torch.dot(image_features[i], image_features[i+1]).item()
                similarities.append(sim)
        else:
            similarities = [0.95, 0.70, 0.92]
            
        current_group = [1]
        for idx, sim in enumerate(similarities):
            frame_id = idx + 2
            if sim < threshold:
                scene_groups.append(current_group)
                current_group = [frame_id]
            else:
                current_group.append(frame_id)
        scene_groups.append(current_group)
        
        return scene_groups, similarities

### 4. 고도화된 물리 법칙 힌트 생성기 (Corrected Panning/Zoom rules)
카메라 수평 패닝(Pan Left/Right)과 피사체 좌우 이동(Move Left/Right)은 X좌표에 미치는 물리적 영향이 정반대이므로, 둘을 엄밀하게 구분하여 안내 텍스트를 빌드합니다.

In [ ]:
def build_3d_spatial_auxiliary_hint(image_paths, sentence, query_text=None):
    aligner = TextEmbeddingAligner()
    w_depth, w_traj, cos_depth, cos_traj = aligner.calculate_weights(sentence)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    extractor = Spatial3DOwlViTTrajectoryExtractor(device=device)
    
    candidate_queries = extract_candidate_nouns(sentence)
    if query_text is not None and query_text not in candidate_queries:
        candidate_queries.append(query_text)
        
    best_query, query_scores = extractor.select_best_query(image_paths, candidate_queries)
    features = extractor.extract_3d_spatial_features(image_paths, best_query)
    
    divider = CLIPSceneDivider(device=device)
    scene_groups, similarities = divider.analyze_cuts(image_paths)
    
    hints = []
    hints.append('[시각 분석 보조 시스템 기하학적 시공간 힌트]')
    hints.append('4장의 뒤섞인 이미지와 자연어 캡션을 대조하여 올바른 프레임 순서를 추론하세요.\n')
    
    hints.append(f'- 캡션 가중치 배분 (임베딩 앵커 코사인 정렬):')
    hints.append(f'  * 깊이 방향 유사도 (Depth Match): {cos_depth:.2f}')
    hints.append(f'  * 평면 궤적 유사도 (Trajectory Match): {cos_traj:.2f}')
    
    recom_axis = '깊이 변화 흐름[주인공의 화면 점유 비율 변화]' if w_depth > w_traj else '평면 궤적 변화[좌우/상하 이동]'
    hints.append(f'  * (시스템 추천: {recom_axis}에 더 큰 가중치를 두어 순서를 정렬하세요.)\n')
    
    hints.append(f'- 장면 분할 정보 (CLIP 인접 유사도: {[round(s, 2) for s in similarities]}):')
    scene_strs = [f'Group {i+1} [Image ' + ', '.join([str(fid) for fid in g]) + ']' for i, g in enumerate(scene_groups)]
    hints.append(f'  * 검출된 장면 그룹: ' + ' / '.join(scene_strs))
    hints.append(f'  * (동일 장면 그룹 내에서만 연속적인 움직임이나 줌 변화 추세를 관찰해야 오판을 막을 수 있습니다.)\n')
    
    hints.append(f'- 핵심 피사체 쿼리 자동 채택 (OWL-ViT):')
    hints.append(f'  * 추출된 캡션 단어 후보군: {candidate_queries}')
    hints.append(f'  * 최종 채택된 쿼리: \'{best_query}\' (평균 검출 신뢰도: {query_scores[best_query]:.3f})')
    hints.append(f'  * 쿼리별 신뢰도 순위: {sorted(query_scores.items(), key=lambda x: x[1], reverse=True)}\n')
    
    hints.append(f'- 물리 측정 데이터 (OWL-ViT - Target: \'{best_query}\'):')
    
    valid_frames = []
    for idx, f in enumerate(features):
        frame_id = f['frame']
        if f['status'] == 'success':
            cx, cy = f['center']
            area = f['area_ratio'] * 100
            hints.append(f'  * Image {frame_id}: \'{best_query}\' center=[X={cx:.3f}, Y={cy:.3f}], Area={area:.1f}%')
            valid_frames.append((frame_id, area))
        else:
            hints.append(f'  * Image {frame_id}: \'{best_query}\' - no object detected (skip this cue)')
            
    if len(valid_frames) >= 2:
        sorted_frames = sorted(valid_frames, key=lambda x: x[1])
        hints.append(f'\n[최종 정렬 가이드]')
        
        # 1. 줌 변화 가이드
        if w_depth > w_traj:
            if len(scene_groups) > 1:
                hints.append(f'주의: 여러 장면 그룹({len(scene_groups)}개)이 감지되었습니다.')
                for i, group in enumerate(scene_groups):
                    group_valid = [vf for vf in valid_frames if vf[0] in group]
                    if len(group_valid) >= 2:
                        sorted_group = sorted(group_valid, key=lambda x: x[1])
                        g_flow = ' -> '.join([f'Image {fid} ({val:.1f}%)' for fid, val in sorted_group])
                        hints.append(f'  * {scene_strs[i]} 내부 면적 흐름: {g_flow}')
            else:
                flow_str = ' -> '.join([f'Image {fid} ({val:.1f}%)' for fid, val in sorted_frames])
                hints.append(f'  * 전체 이미지 면적 흐름: {flow_str}')
            
        # 2. 정교화된 패닝/좌우 이동 인과성 물리 가이드
        s_lower = sentence.lower()
        is_camera_pan_left = any(kw in s_lower for kw in ["pan left", "pans left", "panned left", "panning left", "camera moves left", "camera shifts left"])
        is_camera_pan_right = any(kw in s_lower for kw in ["pan right", "pans right", "panned right", "panning right", "camera moves right", "camera shifts right"])
        is_subject_move_left = any(kw in s_lower for kw in ["move left", "moves left", "moving left", "slides left", "skis left", "runs left", "walks left", "glides left"])
        is_subject_move_right = any(kw in s_lower for kw in ["move right", "moves right", "moving right", "slides right", "skis right", "runs right", "walks right", "glides right"])
        
        valid_x = [(f['frame'], f['center'][0]) for f in features if f['status'] == 'success']
        
        if (is_camera_pan_left or is_camera_pan_right or is_subject_move_left or is_subject_move_right) and len(valid_x) >= 2:
            hints.append(f'또한, 캡션에 카메라의 수평 패닝(Pan) 또는 피사체의 좌우 이동 묘사가 감지되었습니다.')
            
            # 카메라 패닝 (역방향 물리 법칙)
            if is_camera_pan_left:
                hints.append(f'  * 카메라 패닝 법칙: 카메라가 왼쪽(Pan Left)으로 회전/이동하므로, 피사체는 상대적으로 화면 내에서 오른쪽(X 좌표 증가)으로 이동해야 합니다.')
            elif is_camera_pan_right:
                hints.append(f'  * 카메라 패닝 법칙: 카메라가 오른쪽(Pan Right)으로 회전/이동하므로, 피사체는 상대적으로 화면 내에서 왼쪽(X 좌표 감소)으로 이동해야 합니다.')
                
            # 피사체 자체 이동 (순방향 물리 법칙)
            if is_subject_move_left:
                hints.append(f'  * 피사체 이동 법칙: 피사체가 왼쪽(Move Left)으로 직접 이동하므로, 화면 내에서 왼쪽(X 좌표 감소)으로 이동해야 합니다.')
            elif is_subject_move_right:
                hints.append(f'  * 피사체 이동 법칙: 피사체가 오른쪽(Move Right)으로 직접 이동하므로, 화면 내에서 오른쪽(X 좌표 증가)으로 이동해야 합니다.')
            
            sorted_x = sorted(valid_x, key=lambda x: x[1])
            x_flow_str = ' -> '.join([f'Image {fid} (X={val:.3f})' for fid, val in sorted_x])
            hints.append(f'  * 검출된 피사체 X좌표 흐름(증가순): {x_flow_str}')
            
        hints.append(f'위의 정보를 바탕으로 결측치나 다른 장면 그룹의 노이즈를 무시하고 인과관계가 일치하도록 정렬하세요.\n')
    hints.append('\n위의 순수 기하학적 수치와 캡션의 전체 맥락을 대조하여 최종 셔플 인덱스 정답 [n, n, n, n]을 도출하세요.')
    
    return '\n'.join(hints)

### 5. holdout_300 정량 평가 셀
새롭게 설계된 기하학적 시공간 물리 규칙을 splits/holdout_300.csv 전체 샘플에 적용하여 정확도를 검증합니다.

In [ ]:
def evaluate_holdout_consistency():
    holdout_csv = "splits/holdout_300.csv"
    image_dir = "snuaichallenge_data/train"
    features_csv = "snu_clip_features.csv"
    
    if not os.path.exists(holdout_csv):
        print("Holdout CSV not found")
        return
        
    df_holdout = pd.read_csv(holdout_csv)
    df_feat = pd.read_csv(features_csv)
    df = pd.merge(df_holdout, df_feat[['Id', 'predicted_scene_cuts']], on='Id', how='left')
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights, box_score_thresh=0.3).to(device)
    model.eval()
    preprocess = weights.transforms()
    
    results = []
    
    for idx, row in df.iterrows():
        sample_id = str(row['Id'])
        sentence = row['Sentence'].lower()
        ans = ast.literal_eval(row['Answer'])
        shuffled_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
        
        scene_cuts = row['predicted_scene_cuts']
        scene_groups = int(scene_cuts + 1) if not pd.isna(scene_cuts) else 4
        
        zoom_in_kw = ["zoom in", "zooms in", "zoomed in", "zooming in", "closer", "shifts close", "moves in"]
        zoom_out_kw = ["zoom out", "zooms out", "zoomed out", "zooming out", "further", "moves out", "receding", "moves away"]
        
        pan_left_kw = ["pan left", "pans left", "panned left", "panning left", "camera moves left", "camera shifts left"]
        pan_right_kw = ["pan right", "pans right", "panned right", "panning right", "camera moves right", "camera shifts right"]
        
        move_left_kw = ["moves left", "moving left", "slides left", "skis left", "runs left", "walks left"]
        move_right_kw = ["moves right", "moving right", "slides right", "skis right", "runs right", "walks right"]

        has_zoom_in = any(kw in sentence for kw in zoom_in_kw)
        has_zoom_out = any(kw in sentence for kw in zoom_out_kw)
        has_pan_left = any(kw in sentence for kw in pan_left_kw)
        has_pan_right = any(kw in sentence for kw in pan_right_kw)
        has_move_left = any(kw in sentence for kw in move_left_kw)
        has_move_right = any(kw in sentence for kw in move_right_kw)

        is_zoom_sample = has_zoom_in or has_zoom_out
        is_pan_sample = has_pan_left or has_pan_right or has_move_left or has_move_right

        if not is_zoom_sample and not is_pan_sample:
            results.append({
                'Id': sample_id, 'Sentence': row['Sentence'], 'Subject': 'person',
                'DetectCount': 0, 'SceneGroups': scene_groups, 'Status': '➖ 해당 없음 (N/A)', 'Reason': 'No spatial keywords',
                'IsPan': False, 'IsZoom': False
            })
            continue
            
        ordered_files = [None] * 4
        for i, pos in enumerate(ans): ordered_files[pos - 1] = shuffled_files[i]
        img_paths = [os.path.join(image_dir, sample_id, f) for f in ordered_files]
        
        centers_x = []; areas = []; detect_count = 0
        for img_path in img_paths:
            img = Image.open(img_path).convert("RGB")
            w, h = img.size
            img_area = w * h
            
            input_tensor = preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad(): predictions = model(input_tensor)[0]
            
            boxes = predictions['boxes'].cpu().numpy()
            max_box_area = 0.0; best_x_center = 0.5
            
            for box in boxes:
                box_w, box_h = box[2] - box[0], box[3] - box[1]
                box_area = (box_w * box_h) / img_area
                if box_area > max_box_area:
                    max_box_area = box_area
                    best_x_center = ((box[0] + box[2]) / 2) / w
            if max_box_area > 0: detect_count += 1
            centers_x.append(best_x_center)
            areas.append(max_box_area)
            
        status = "➖ 해당 없음 (N/A)"; reason = "N/A"
        if detect_count >= 2:
            if is_pan_sample:
                valid_idx = [i for i, a in enumerate(areas) if a > 0]
                x_start = centers_x[valid_idx[0]]
                x_end = centers_x[valid_idx[-1]]
                
                expected_right = has_pan_left or has_move_right
                expected_left = has_pan_right or has_move_left
                
                if expected_right and expected_left:
                    status, reason = "➖ 해당 없음 (N/A)", "Mixed pan direction"
                elif expected_right:
                    if x_end > x_start:
                        status = "✅ 일치 (Consistent)"
                        reason = f"Object moved right (X: {x_start:.2f} -> {x_end:.2f})"
                    else:
                        status = "❌ 불일치 (Inconsistent)"
                        reason = f"Object did not move right (X: {x_start:.2f} -> {x_end:.2f})"
                elif expected_left:
                    if x_end < x_start:
                        status = "✅ 일치 (Consistent)"
                        reason = f"Object moved left (X: {x_start:.2f} -> {x_end:.2f})"
                    else:
                        status = "❌ 불일치 (Inconsistent)"
                        reason = f"Object did not move left (X: {x_start:.2f} -> {x_end:.2f})"
                        
            elif is_zoom_sample:
                valid_idx = [i for i, a in enumerate(areas) if a > 0]
                a_start = areas[valid_idx[0]]
                a_end = areas[valid_idx[-1]]
                
                if has_zoom_in and has_zoom_out:
                    status, reason = "➖ 해당 없음 (N/A)", "Mixed zoom direction"
                elif has_zoom_in:
                    if a_end > a_start:
                        status = "✅ 일치 (Consistent)"
                        reason = f"Zoom-in matched area increase ({a_start*100:.1f}% -> {a_end*100:.1f}%)"
                    else:
                        status = "❌ 불일치 (Inconsistent)"
                        reason = f"Zoom-in failed (area decreased: {a_start*100:.1f}% -> {a_end*100:.1f}%)"
                elif has_zoom_out:
                    if a_end < a_start:
                        status = "✅ 일치 (Consistent)"
                        reason = f"Zoom-out matched area decrease ({a_start*100:.1f}% -> {a_end*100:.1f}%)"
                    else:
                        status = "❌ 불일치 (Inconsistent)"
                        reason = f"Zoom-out failed (area increased: {a_start*100:.1f}% -> {a_end*100:.1f}%)"
                        
        results.append({
            'Id': sample_id, 'Sentence': row['Sentence'], 'Subject': 'person',
            'DetectCount': detect_count, 'SceneGroups': scene_groups,
            'Status': status, 'Reason': reason, 'IsPan': is_pan_sample, 'IsZoom': is_zoom_sample
        })
        
    res_df = pd.DataFrame(results)
    
    # 결과 요약 출력
    pan_df = res_df[res_df['IsPan']]
    zoom_df = res_df[res_df['IsZoom'] & ~res_df['IsPan']]
    pan_valid = pan_df[pan_df['Status'] != "➖ 해당 없음 (N/A)"]
    zoom_valid = zoom_df[zoom_df['Status'] != "➖ 해당 없음 (N/A)"]
    
    pan_ok = sum(pan_valid['Status'].str.contains("✅"))
    zoom_ok = sum(zoom_valid['Status'].str.contains("✅"))
    
    print("\n" + "="*50)
    print("📊 종합 요약 지표 (Summary Metrics)")
    print("="*50)
    print(f"* 평가 대상 샘플 수: {len(df)}개")
    print(f"* 평균 객체 검출 수: {res_df['DetectCount'].mean():.2f} / 4 프레임")
    print(f"* 평균 장면 그룹 수 (CLIP): {res_df['SceneGroups'].mean():.2f}개")
    print(f"* 물리적 정합성 검증 대상 샘플 수: {len(pan_valid)+len(zoom_valid)}개")
    print(f"* 전체 물리 법칙 정합률: {(pan_ok+zoom_ok)/(len(pan_valid)+len(zoom_valid))*100:.1f}% ({pan_ok+zoom_ok}/{len(pan_valid)+len(zoom_valid)})")
    print(f"  - 카메라 줌(Zoom-in/out) 정합률: {zoom_ok/len(zoom_valid)*100:.1f}% ({zoom_ok}/{len(zoom_valid)})")
    print(f"  - 카메라 패닝(Pan-left/right) 및 좌우이동 정합률: {pan_ok/len(pan_valid)*100:.1f}% ({pan_ok}/{len(pan_valid)})")
    
    # 표 형태로 출력
    print("\n" + "="*120)
    print(f"{'샘플 Id':<10} | {'채택 피사체':<8} | {'검출 프레임수':<8} | {'장면그룹수':<6} | {'물리 정합성 여부':<12} | {'정합성 판별 근거'}")
    print("="*120)
    for _, row in res_df.iterrows():
        print(f"{row['Id']:<10} | {row['Subject']:<8} | {row['DetectCount']:<8} | {row['SceneGroups']:<6} | {row['Status']:<12} | {row['Reason']}")

evaluate_holdout_consistency()